In [1]:
## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

In [9]:
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py

results_iteration = healthiar.attribute_health(
# Names of Swiss cantons
    geo_id_micro = py_to_r(["Zurich", "Basel", "Geneva", "Ticino", "Jura"]),
    # Names of languages spoken in the selected Swiss cantons
    geo_id_macro = py_to_r(["German","German","French","Italian","French"]),
    rr_central = 1.369,
    rr_increment = 10, 
    cutoff_central = 5,
    erf_shape = "log_linear",
    exp_central = py_to_r([11, 11, 10, 8, 7]),
    bhd_central = py_to_r([4000, 2500, 3000, 1500, 500])
)

py_results_iteration = r_to_py(results_iteration)
py_results_iteration["health_main"][["geo_id_macro", "impact_rounded", "erf_ci", "exp_ci", "bhd_ci"]].head()

,geo_id_macro,impact_rounded,erf_ci,exp_ci,bhd_ci
1,German,1116.0,central,central,central
2,French,466.0,central,central,central
3,Italian,135.0,central,central,central


In [10]:
import rpy2.rinterface as ri
from scipy.interpolate import CubicSpline

## define ERF with decorator ('@')
@ri.rternalize
def erf_fun(x):
    cs = CubicSpline(
      x = [0, 5, 10, 15, 20, 25, 30, 50, 70, 90, 110],
      y = [1.00, 1.04, 1.08, 1.12, 1.16, 1.20, 1.23, 1.35, 1.45, 1.53, 1.60]
    )
    return float(cs(x)[0])

## pass ERF to attribute_health()
results_pm_copd_mr_brt = healthiar.attribute_health(
  exp_central = 8.85,
  bhd_central = 30747,
  cutoff_central = 0,
  erf_eq_central = erf_fun
)

In [15]:
from healthiar.spatial import read_raster, read_vector
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows

path = files("healthiar.data")
exdat_pwm_1 = read_raster(path.joinpath("pm25.tif").as_posix())
exdat_pwm_2 = read_vector(path.joinpath("municipalities_brussels.gpkg").as_posix(), quiet = True)

population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]

geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)

#py_pwm = r_to_py(pwm)
#print(py_pwm["exposure_main"])